In [4]:
import json

In [7]:
import pandas as pd

graded_df = pd.read_json("../results/raw/p2_pilot_responses.jsonl", lines=True)
pilot = pd.read_csv("../data/manifests/p2_pilot_200.csv")


In [8]:
graded_df = graded_df.merge(pilot[["question_id", "problem"]], on="question_id", how="left")
graded_df

,question_id,sample_index,is_greedy,generation_seed,model_answer,answer,outcome,problem
0,4,0,True,NaN,Karl Küchler is known for his work on the Vill...,Poet Henrich Heine.,incorrect,"According to Karl Küchler, what did Empress El..."
1,4,1,False,4001.0,Karl Küchler is known for his interpretations ...,Poet Henrich Heine.,incorrect,"According to Karl Küchler, what did Empress El..."
2,4,2,False,4002.0,Karl Küchler is known for his art criticism on...,Poet Henrich Heine.,incorrect,"According to Karl Küchler, what did Empress El..."
3,4,3,False,4003.0,Karl Küchler is known for his work on the reco...,Poet Henrich Heine.,incorrect,"According to Karl Küchler, what did Empress El..."
4,4,4,False,4004.0,Karl Küchler is known for his paintings of his...,Poet Henrich Heine.,incorrect,"According to Karl Küchler, what did Empress El..."
...,...,...,...,...,...,...,...,...
1195,4312,1,False,4312001.0,Baleka Mbete became the Deputy President of So...,2008,incorrect,In what year did Baleka Mbete become the Deput...
1196,4312,2,False,4312002.0,Baleka Mbete became the Deputy President of So...,2008,incorrect,In what year did Baleka Mbete become the Deput...
1197,4312,3,False,4312003.0,Baleka Mbete became the Deputy President of So...,2008,incorrect,In what year did Baleka Mbete become the Deput...
1198,4312,4,False,4312004.0,Baleka Mbete became the Deputy President of So...,2008,incorrect,In what year did Baleka Mbete become the Deput...


In [18]:
print(len(graded_df))

1200


In [13]:
n_per_outcome = 40
counts = graded_df["outcome"].value_counts()
counts

outcome
incorrect    1011
abstain       172
correct        17
Name: count, dtype: int64

In [14]:
sample_sizes = counts.clip(upper=n_per_outcome)  
sample_sizes

outcome
incorrect    40
abstain      40
correct      17
Name: count, dtype: int64

In [15]:
audit_sample = pd.concat([
    graded_df[graded_df["outcome"] == outcome].sample(n=sample_sizes[outcome], random_state=1)
    for outcome in counts.index
]).reset_index(drop=True)
audit_sample

,question_id,sample_index,is_greedy,generation_seed,model_answer,answer,outcome,problem
0,3956,1,False,3956001.0,The developmental code for Firazorexton is: FE...,TAK-994,incorrect,What is the developmental code for Firazorexto...
1,3082,5,False,3082005.0,The bronze medal was won by British boxer Geor...,No one.,incorrect,Who won the bronze medal in the men's masters ...
2,1301,1,False,1301001.0,"The domain ""hindustantimes.com"" was registered...","August 14, 1996",incorrect,"On which day, month, and year was the domain ""..."
3,2333,1,False,2333001.0,Ken Noda was invited by President Ronald Reaga...,20,incorrect,At what age was Ken Noda invited by President ...
4,4168,0,True,NaN,Dr. Abu Baker Aswat played for the Pakistan Cr...,The Crescents,incorrect,What was the name of the cricket team Dr. Abu ...
...,...,...,...,...,...,...,...,...
92,3210,1,False,3210001.0,There are only about 10-20 African forest elep...,20,correct,What is the maximum number of African forest e...
93,2411,1,False,2411001.0,Mowalola Ogunlesi dropped out of Central Saint...,2018,correct,In what year did fashion designer Mowalola Ogu...
94,3511,2,False,3511002.0,"According to current records as of 2023, there...",3,correct,How many amendments to the Pakistani Constitut...
95,3511,0,True,NaN,"As of 2022, there are no amendments to Pakista...",3,correct,How many amendments to the Pakistani Constitut...


In [16]:
print(audit_sample["outcome"].value_counts())
print(f"Total selected for audit: {len(audit_sample)}")

outcome
incorrect    40
abstain      40
correct      17
Name: count, dtype: int64
Total selected for audit: 97


In [17]:
audit_sample["human_label"] = ""
audit_sample["notes"] = ""
valid_labels = {"correct", "incorrect", "abstain"}
i = 0
for idx, row in audit_sample.iterrows():
    print("=" * 60)
    print(f"Row {idx} | question_id={row['question_id']} | heuristic outcome={row['outcome']}")
    print(f"Question:        {row['problem']}")
    print(f"Reference answer: {row['answer']}")
    print(f"Model answer:     {row['model_answer']}")
    i += 1
    if i > 5:
        break

Row 0 | question_id=3956 | heuristic outcome=incorrect
Question:        What is the developmental code for Firazorexton, an orally active, brain-permeable orexin type 2 receptor agonist?
Reference answer:  TAK-994
Model answer:     The developmental code for Firazorexton is: FERZ-014.
Row 1 | question_id=3082 | heuristic outcome=incorrect
Question:        Who won the bronze medal in the men's masters foil event in the 1896 Summer Olympics?
Reference answer: No one.
Model answer:     The bronze medal was won by British boxer George Haines in the men's masters foil event at the 1896 Summer Olympics held in Athens, Greece.
Row 2 | question_id=1301 | heuristic outcome=incorrect
Question:        On which day, month, and year was the domain "hindustantimes.com" registered?
Reference answer: August 14, 1996
Model answer:     The domain "hindustantimes.com" was registered on December 10, 2019.
Row 3 | question_id=2333 | heuristic outcome=incorrect
Question:        At what age was Ken Noda invi

In [3]:
import pandas as pd
import pipeline  # needed for pipeline.normalize()

graded_df = pd.read_json("../results/raw/p2_pilot_responses.jsonl", lines=True)
pilot = pd.read_csv("../data/manifests/p2_pilot_200.csv")
graded_df = graded_df.merge(pilot[["question_id", "problem"]], on="question_id", how="left")

def partial_overlap(model_answer, reference_answer):
    ref_words = set(pipeline.normalize(reference_answer).split())
    ans_words = set(pipeline.normalize(model_answer).split())
    return len(ref_words & ans_words) > 0 and len(ref_words) > 0

incorrect_rows = graded_df[graded_df["outcome"] == "incorrect"]
suspicious = incorrect_rows[
    incorrect_rows.apply(lambda r: partial_overlap(r["model_answer"], r["answer"]), axis=1)
]
print(f"{len(suspicious)} 'incorrect' rows share at least one word with the reference — check these first")
suspicious[["question_id", "problem", "answer", "model_answer", "outcome"]].head(20)

210 'incorrect' rows share at least one word with the reference — check these first


,question_id,problem,answer,model_answer,outcome
18,145,Who was the architect of Leamington who design...,John Cundall,The architect of Leamington who designed the c...,incorrect
19,145,Who was the architect of Leamington who design...,John Cundall,The architect of Leamington who designed the c...,incorrect
20,145,Who was the architect of Leamington who design...,John Cundall,The architect of Leamington who designed the c...,incorrect
21,145,Who was the architect of Leamington who design...,John Cundall,The architect of Leamington who designed the c...,incorrect
22,145,Who was the architect of Leamington who design...,John Cundall,The architect of Leamington who designed the c...,incorrect
23,145,Who was the architect of Leamington who design...,John Cundall,The architect of Leamington that designed the ...,incorrect
60,292,"As a teen, what job sparked Shonda Rhimes' int...",hospital volunteer,Shonda Rhimes' interest in hospital environmen...,incorrect
61,292,"As a teen, what job sparked Shonda Rhimes' int...",hospital volunteer,Shonda Rhimes's interest in hospital environme...,incorrect
62,292,"As a teen, what job sparked Shonda Rhimes' int...",hospital volunteer,Shonda Rhimes' interest in hospital environmen...,incorrect
63,292,"As a teen, what job sparked Shonda Rhimes' int...",hospital volunteer,Shonda Rhimes' interest in hospital environmen...,incorrect


In [4]:
suspicious.to_csv('../results/audit/suspicious.csv', index=False)

In [5]:
k_scores = pd.read_csv("../results/features/p2_pilot_k_scores.csv")


n_k0 = (k_scores["K"] == 0).sum()
n_mid = ((k_scores["K"] > 0) & (k_scores["K"] < 1)).sum()
n_high = (k_scores["K"] >= 0.8).sum()
print(f"K=0: {n_k0}, 0<K<1: {n_mid}, K>=0.8: {n_high}")

K=0: 192, 0<K<1: 7, K>=0.8: 1
